# Fine-Tuning with PEFT (LoRA)

## Overview

This notebook teaches you how to fine-tune any HuggingFace model using LoRA (Low-Rank Adaptation).

**What you'll do:**
1. Load a pre-trained model
2. Apply LoRA to make it trainable with limited GPU memory
3. Train on your custom dataset
4. Evaluate and save the trained adapter

**Key advantage:** Train with 1/10th the GPU memory and time of full fine-tuning!

## ⚙️ Configuration

**Modify these values for your use case:**

In [ ]:
# ============================================
# MODEL CONFIGURATION
# ============================================

# Path to base model (HuggingFace ID or local path)
MODEL_NAME = "/mnt/models/meta-llama/Llama-2-7b-hf"  # or "meta-llama/Llama-2-7b" for HF Hub
MODEL_DTYPE = "bfloat16"  # or "float16", "float32"
DEVICE_MAP = "auto"  # Automatically place on available GPUs

# ============================================
# DATA CONFIGURATION
# ============================================

# Path to training data (CSV or JSON)
TRAIN_DATA_PATH = "/mnt/datasets/train.csv"
VAL_DATA_PATH = "/mnt/datasets/val.csv"  # Optional

# Data column names (adjust if your data has different columns)
TEXT_COLUMN = "text"  # or "instruction"
LABEL_COLUMN = "label"  # or "output"
MAX_LENGTH = 512  # Max sequence length (adjust for longer/shorter texts)

# ============================================
# LoRA CONFIGURATION
# ============================================

# LoRA parameters
LORA_R = 8  # Rank of LoRA matrices (4-32 typical; higher = more capacity)
LORA_ALPHA = 16  # Scaling factor (usually 2x rank)
LORA_DROPOUT = 0.05  # Dropout for regularization
TARGET_MODULES = ["q_proj", "v_proj"]  # Which layers to apply LoRA to
BIAS = "none"  # How to handle bias ("none", "all", or "lora_only")
TASK_TYPE = "CAUSAL_LM"  # Type of task (CAUSAL_LM, SEQ_2_SEQ_LM, etc.)

# ============================================
# TRAINING CONFIGURATION
# ============================================

# Training hyperparameters
LEARNING_RATE = 3e-4  # Typical: 1e-5 to 1e-3
BATCH_SIZE = 8  # Adjust based on GPU memory (8-16 typical for LoRA)
NUM_EPOCHS = 3  # Number of training passes through data
WARMUP_STEPS = 100  # Learning rate warmup
WEIGHT_DECAY = 0.01  # L2 regularization
MAX_GRAD_NORM = 1.0  # Gradient clipping

# ============================================
# OUTPUT CONFIGURATION
# ============================================

OUTPUT_DIR = "/mnt/outputs"  # Where to save adapter
ADAPTER_NAME = "lora-finetuned"  # Name for your adapter
SAVE_STEPS = 100  # Save checkpoint every N steps
EVAL_STEPS = 100  # Evaluate every N steps

## 📦 Step 1: Install & Import Libraries

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path

import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from datasets import Dataset

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA device: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 📥 Step 2: Load Model and Tokenizer

In [ ]:
logger.info(f"Loading model: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
logger.info(f"Tokenizer loaded (vocab size: {len(tokenizer)})")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if MODEL_DTYPE == "bfloat16" else torch.float16,
    device_map=DEVICE_MAP,
)
logger.info(f"Model loaded")

# Print model size
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Total parameters: {total_params / 1e9:.2f}B")

## 🎯 Step 3: Apply LoRA Configuration

In [ ]:
# Define LoRA configuration
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias=BIAS,
    task_type=TASK_TYPE,
)

logger.info(f"LoRA Config:")
logger.info(f"  Rank (r): {LORA_R}")
logger.info(f"  Alpha: {LORA_ALPHA}")
logger.info(f"  Target modules: {TARGET_MODULES}")
logger.info(f"  Dropout: {LORA_DROPOUT}")

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percent_trainable = (trainable_params / total_params) * 100

logger.info(f"\nTrainable parameters: {trainable_params / 1e6:.2f}M / {total_params / 1e9:.2f}B ({percent_trainable:.2f}%)")
logger.info(f"Model is ready for training!")

## 📊 Step 4: Load and Prepare Training Data

In [ ]:
def load_dataset(file_path, text_col, label_col):
    """Load data from CSV or JSON"""
    logger.info(f"Loading data from: {file_path}")
    
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith('.json'):
        df = pd.read_json(file_path)
    else:
        raise ValueError(f"Unsupported format: {file_path}")
    
    logger.info(f"Loaded {len(df)} examples")
    logger.info(f"Columns: {list(df.columns)}")
    
    # Check required columns exist
    if text_col not in df.columns or label_col not in df.columns:
        raise ValueError(f"Columns {text_col}, {label_col} not found in data")
    
    return df

# Load training data
train_df = load_dataset(TRAIN_DATA_PATH, TEXT_COLUMN, LABEL_COLUMN)

# Load validation data if available
val_df = None
if Path(VAL_DATA_PATH).exists():
    val_df = load_dataset(VAL_DATA_PATH, TEXT_COLUMN, LABEL_COLUMN)
else:
    logger.info(f"Validation data not found, will use 10% of training data")
    # Split training data
    val_size = max(1, len(train_df) // 10)
    val_df = train_df.iloc[:val_size].reset_index(drop=True)
    train_df = train_df.iloc[val_size:].reset_index(drop=True)
    logger.info(f"Split: {len(train_df)} train, {len(val_df)} validation")

# Display sample
logger.info(f"\nSample from training data:")
print(train_df.iloc[0])

## 🔄 Step 5: Create Data Processing Function

In [ ]:
def preprocess_function(examples):
    """Tokenize and prepare examples for training"""
    # Concatenate text and label
    texts = [f"{example[TEXT_COLUMN]} {example[LABEL_COLUMN]}" 
             for example in examples]
    
    # Tokenize
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    
    # For causal LM, labels are the input IDs shifted
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# Convert to HuggingFace Dataset
def df_to_dataset(df):
    # Convert to dict format expected by Dataset
    data_dict = {
        TEXT_COLUMN: df[TEXT_COLUMN].tolist(),
        LABEL_COLUMN: df[LABEL_COLUMN].tolist(),
    }
    dataset = Dataset.from_dict(data_dict)
    
    # Apply preprocessing
    dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing",
    )
    return dataset

logger.info("Preparing datasets...")
train_dataset = df_to_dataset(train_df)
val_dataset = df_to_dataset(val_df) if val_df is not None else None

logger.info(f"Training examples: {len(train_dataset)}")
if val_dataset:
    logger.info(f"Validation examples: {len(val_dataset)}")

## 🚀 Step 6: Setup Training

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    evaluation_strategy="steps" if val_dataset else "no",
    save_strategy="steps",
    fp16=False,  # Use bfloat16 instead
    bf16=True,
    gradient_accumulation_steps=1,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to=[],  # Disable wandb/tensorboard
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

logger.info(f"Training setup complete. Starting training...")

## 📈 Step 7: Train the Model

In [ ]:
import time

start_time = time.time()
logger.info(f"Starting training...")

# Train
train_result = trainer.train()

elapsed_time = (time.time() - start_time) / 3600  # Convert to hours
logger.info(f"Training completed in {elapsed_time:.2f} hours")
logger.info(f"Final training loss: {train_result.training_loss:.4f}")

## ✅ Step 8: Evaluate

In [ ]:
if val_dataset:
    logger.info("Evaluating on validation set...")
    eval_results = trainer.evaluate()
    
    logger.info(f"\nEvaluation Results:")
    for key, value in eval_results.items():
        logger.info(f"  {key}: {value}")
else:
    eval_results = {}

## 💾 Step 9: Save Adapter and Metrics

In [ ]:
# Save adapter
adapter_path = Path(OUTPUT_DIR) / ADAPTER_NAME
adapter_path.mkdir(parents=True, exist_ok=True)

logger.info(f"Saving adapter to {adapter_path}")
model.save_pretrained(str(adapter_path))

# Save tokenizer
tokenizer.save_pretrained(str(adapter_path))

# Save metrics
metrics = {
    "model_name": MODEL_NAME,
    "training_loss": float(train_result.training_loss),
    "evaluation_results": eval_results,
    "lora_config": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "target_modules": TARGET_MODULES,
    },
    "training_time_hours": elapsed_time,
    "training_samples": len(train_dataset),
    "validation_samples": len(val_dataset) if val_dataset else 0,
}

metrics_path = Path(OUTPUT_DIR) / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

logger.info(f"Metrics saved to {metrics_path}")
logger.info(f"\n✅ Fine-tuning complete!")
logger.info(f"Adapter saved to: {adapter_path}")

## 🧪 Step 10: Test Inference (Optional)

Generate some text with your fine-tuned model:

In [ ]:
# Example inference
def generate_text(prompt, max_length=100):
    """Generate text using the fine-tuned model"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with a sample prompt
if len(train_df) > 0:
    sample_prompt = train_df.iloc[0][TEXT_COLUMN]
    logger.info(f"\nTesting inference with prompt:")
    logger.info(f"Prompt: {sample_prompt}")
    
    output = generate_text(sample_prompt, max_length=200)
    logger.info(f"\nGenerated output:")
    logger.info(output)